In [ ]:
import xml.etree.ElementTree as et
#from lxml import etree
import pandas as pd
from IPython.core import display as ICD
from pandas import json_normalize
import pandas as pd
#import tqdm as tqdm
from tqdm.notebook import tqdm
import os
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
from datetime import datetime
import time



#@title
types_dict={1:"Pass",2:"Offside Pass",3:"Take On",4:"Foul",5:"Out",6:"Corner Awarded",7:"Tackle",8:"Interception",9:"Turnover",10:"Save",11:"Claim",12:"Clearance",13:"Miss",14:"Post",15:"Attempt Saved",16:"Goal",17:"Card",18:"Player off",19:"Player on",20:"Player retired",21:"Player returns",22:"Player becomes goalkeeper",23:"Goalkeeper becomes player",24:"Condition change",25:"Official change",27:"Start delay",28:"End delay",30:"End",31:"Picked an orange",32:"Start",34:"Team set up",35:"Player changed position",36:"Player changed Jersey number",37:"Collection End",38:"Temp_Goal",39:"Temp_Attempt",40:"Formation change",41:"Punch",42:"Good Skill",43:"Deleted event",44:"Aerial",45:"Challenge",47:"Rescinded card",49:"Ball recovery",50:"Dispossessed",51:"Error",52:"Keeper pick-up",53:"Cross not claimed",54:"Smother",55:"Offside provoked",56:"Shield ball opp",57:"Foul throw-in",58:"Penalty faced",59:"Keeper Sweeper",60:"Chance missed",61:"Ball touch",63:"Temp_Save",64:"Resume",65:"Contentious referee decision",67:"50/50",68:"Referee drop ball",69:"Failed To Block",72:"Caught offside",73:"Other Ball Contact",74:"Blocked pass"};
types = pd.DataFrame.from_dict(types_dict,orient='index').reset_index()
types.columns=["type_id","event_name"]

qualifiers_dict={1:"Long ball",2:"Cross",3:"Head pass",4:"Through ball",5:"Free kick taken",6:"Corner taken",7:"Players caught offside",8:"Goal disallowed",9:"Penalty",10:"Hand",11:"6-seconds violation",12:"Dangerous play",13:"Foul",14:"Last line",15:"Head",16:"Small box-centre",17:"Box-centre",18:"Out of box-centre",19:"35+ centre",20:"Right footed",21:"Other body part",22:"Regular play",23:"Fast break",24:"Set piece",25:"From corner",26:"Free kick",28:"Own goal",29:"Assisted",30:"Involved",31:"Yellow Card",32:"Second yellow",33:"Red card",34:"Referee abuse",35:"Argument",36:"Fight",37:"Time wasting",38:"Excessive celebration",39:"Crowd interaction",40:"Other reason",41:"Injury",42:"Tactical",44:"Player position",49:"Attendance figure",50:"Official position",51:"Official ID",53:"Injured player id",54:"End cause",55:"Related event ID",56:"Zone",57:"End type",59:"Jersey number",60:"Small box-right",61:"Small box-left",62:"Box-deep right",63:"Box-right",64:"Box-left",65:"Box-deep left",66:"Out of box-deep right",67:"Out of box-right",68:"Out of box-left",69:"Out of box-deep left",70:"35+ right",71:"35+ left",72:"Left footed",73:"Left",74:"High",75:"Right",76:"Low left",77:"High left",78:"Low centre",79:"High centre",80:"Low right",81:"High right",82:"Blocked",83:"Close left",84:"Close right",85:"Close high",86:"Close left and high",87:"Close right and high",88:"High claim",89:"1 on 1",90:"Deflected save",91:"Dive and deflect",92:"Catch",93:"Dive and catch",94:"Def block",95:"Back pass",96:"Corner situation",97:"Direct free",100:"Six yard blocked",101:"Saved off line",102:"Goal mouth y co-ordinate",103:"Goal mouth z co-ordinate",106:"Attacking Pass",107:"Throw-in",108:"Volley",109:"Overhead",110:"Half Volley",111:"Diving Header",112:"Scramble",113:"Strong",114:"Weak",115:"Rising",116:"Dipping",117:"Lob",118:"One Bounce",119:"Few Bounces",120:"Swerve Left",121:"Swerve Right",122:"Swerve Moving",123:"Keeper Throw",124:"Goal Kick",127:"Direction of play",128:"Punch",130:"Team formation",131:"Team player formation",132:"Dive",133:"Deflection",134:"Far Wide Left",135:"Far Wide Right",136:"Keeper Touched",137:"Keeper Saved",138:"Hit Woodwork",139:"Own Player",140:"Pass End X",141:"Pass End Y",144:"Deleted event type",145:"Formation slot",146:"Blocked x co-ordinate",147:"Blocked y co-ordinate",153:"Not past goal line",154:"Intentional assist",155:"Chipped",156:"Lay-off",157:"Launch",158:"Persistent infringement",159:"Foul and abusive language",160:"Throw-in set piece",161:"Encroachment",162:"Leaving field",163:"Entering field",164:"Spitting",165:"Professional foul",166:"Handling on the line",167:"Out of play",168:"Flick-on",169:"Leading to attempt",170:"Leading to goal",171:"Rescinded card",172:"No impact on timing",173:"Parried safe",174:"Parried danger",175:"Fingertip",176:"Caught",177:"Collected",178:"Standing",179:"Diving",180:"Stooping",181:"Reaching",182:"Hands",183:"Feet",184:"Dissent",185:"Blocked cross",186:"Scored",187:"Saved",188:"Missed",189:"Player not visible",190:"From shot off target",191:"Off the ball foul",192:"Block by hand",194:"Captain",195:"Pull Back",196:"Switch of play",197:"Team kit",198:"GK hoof",199:"Gk kick from hands",200:"Referee stop",201:"Referee delay",202:"Weather problem",203:"Crowd trouble",204:"Fire",205:"Object thrown on pitch",206:"Spectator on pitch",207:"Awaiting officials decision",208:"Referee Injury",209:"Game end",210:"Assist",211:"Overrun",212:"Length",213:"Angle",214:"Big Chance",215:"Individual Play",216:"2nd related event ID",217:"2nd assisted",218:"2nd assist",219:"Players on both posts",220:"Player on near post",221:"Player on far post",222:"No players on posts",223:"In-swinger",224:"Out-swinger",225:"Straight",226:"Suspended",227:"Resume",228:"Own shot blocked",229:"Post-match complete",230:"GK X Coordinate",231:"GK Y Coordinate",232:"Unchallenged"};
qualifiers = pd.DataFrame.from_dict(qualifiers_dict,orient='index').reset_index()
qualifiers.columns = ["qualifier_id","description"]

qualifiers_dict2 = {str(key): str(value) for key, value in qualifiers_dict.items()}


def parsef24_folder(F24folder):
  games_list = []
  events_list = []

  for file in tqdm(os.listdir(F24folder)):
    if file.endswith(".xml"):
      file_path = os.path.join(F24folder, file)
      #print(f"Processing: {file_path}")

      tree = et.ElementTree(file=file_path)
      games = tree.getroot()
      gameinfo = games.findall('Game')[0]  # Assuming there's always one 'Game' element

      # Cache game metadata
      game_id = gameinfo.get('id')
      game_meta = {
          "game_id": game_id,
          "home_team_id": gameinfo.get('home_team_id'),
          "home_team_name": gameinfo.get('home_team_name'),
          "away_team_id": gameinfo.get('away_team_id'),
          "away_team_name": gameinfo.get('away_team_name'),
          "competition_id": gameinfo.get('competition_id'),
          "competition_name": gameinfo.get('competition_name'),
          "season_id": gameinfo.get('season_id'),
      }
      games_list.append(game_meta)

      for game in games:
        for event in game:
          # Build a dictionary for the event data
          event_data = event.attrib.copy()
          # Use list comprehension to extract qualifiers
          event_data["qualifiers"] = [q.attrib for q in event]
          event_data["game_id"] = game_id  # Attach game metadata to event
          # Build a DataFrame for this event and append it to the list
          events_list.append(event_data)

  # Concatenate all parsed events into a single DataFrame
  game_df = pd.DataFrame(games_list)
  match_events = pd.DataFrame(events_list)

  match_events[["id","event_id","type_id","period_id","min","sec"]] = match_events[["id","event_id","type_id","period_id","min","sec"]].astype(int)
  match_events[["y","x"]] = match_events[["y","x"]].astype(float)
  match_events = pd.merge(match_events,types, on="type_id", how = "left")
  match_events = match_events[ ['id',"event_id","type_id", "event_name" ]+ [ col for col in match_events.columns if col not in ['id',"event_id","type_id", "event_name" ] ] ]

  # add game info to match_events
  match_events = pd.merge(match_events, game_df, on="game_id", how="inner")

  return match_events


def explode_event(nome_df, id_evento, mytresh):
    # Filter the dataframe for the required event type
    nome_df = nome_df[nome_df["type_id"] == id_evento].copy()

    if nome_df.empty:
        return pd.DataFrame()  # Return empty if no matching events

    # Explode 'qualifiers' column (assuming it's a list of dictionaries)
    nome_df_exploded = nome_df.explode("qualifiers")

    # Normalize the qualifiers column
    qualifiers_df = pd.json_normalize(nome_df_exploded["qualifiers"]).fillna("yes")

    # Add the event ID back to qualifiers_df
    qualifiers_df["id"] = nome_df_exploded["id"].values

    # pivot table
    qualifiers_df = qualifiers_df\
      .pivot_table(index='id', columns='qualifier_id', values='value', aggfunc='first')\
      .reset_index()

    # Rename columns based on your dictionary
    qualifiers_df.rename(columns=qualifiers_dict2, inplace=True)

    # Drop columns that have too many NaN values
    min_non_na = len(qualifiers_df) * mytresh
    qualifiers_df = qualifiers_df.dropna(thresh=min_non_na, axis=1)

    # Drop the original exploded 'qualifiers' column
    nome_df = nome_df.drop(columns=["qualifiers"])

    # Merge back
    exploded_df = nome_df.merge(qualifiers_df, on="id", how="outer").fillna("-")

    return exploded_df

#**PLEASE READ**:


Run the cell above, which contains the function code, before proceeding


#**GLOSSARY**:


**types** - DataFrame containing all event type IDs and their corresponding names.
Print `types.head()` to get an idea of its contents.


**qualifiers** - DataFrame containing all qualifier IDs and their corresponding names.
Print `qualifiers.head()` to get an idea of its contents.



---







**parsef24_folder** - Function that converts Opta F24 XML feeds into a pandas DataFrame.
To run correctly, it must receive as a `parameter` the folder where the XML files to be converted are stored (the folder must already be uploaded to Google Colab).



---




**explode_event** - Function that takes the `match_events` DataFrame (or another one already filtered by you) and the event type you want, and “explodes” all qualifiers into columns.
The last parameter is the threshold that defines the minimum percentage of non-null values that columns must contain.
If it is set to 1, it will return only columns with no NaN values; if it is set to 0, it will return all columns.

Example:
```
explode_event(match_events, 1, 0.15)
```

In [134]:
qualifiers.head()

,qualifier_id,description
0,1,Long ball
1,2,Cross
2,3,Head pass
3,4,Through ball
4,5,Free kick taken


In [135]:
match_events = parsef24_folder("/content/OPTA")

  0%|          | 0/380 [00:00<?, ?it/s]

In [136]:
#Working on the Data Type
match_events.info()
match_events["outcome"] = match_events["outcome"].astype(int)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 660881 entries, 0 to 660880
Data columns (total 27 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                660881 non-null  int64  
 1   event_id          660881 non-null  int64  
 2   type_id           660881 non-null  int64  
 3   event_name        655470 non-null  object 
 4   period_id         660881 non-null  int64  
 5   min               660881 non-null  int64  
 6   sec               660881 non-null  int64  
 7   team_id           660881 non-null  object 
 8   outcome           660881 non-null  object 
 9   x                 660881 non-null  float64
 10  y                 660881 non-null  float64
 11  timestamp         660881 non-null  object 
 12  timestamp_utc     660881 non-null  object 
 13  last_modified     660881 non-null  object 
 14  version           660881 non-null  object 
 15  qualifiers        660881 non-null  object 
 16  game_id           66

In [180]:
#Visualize the Data
match_events.sample(5)
match_events.groupby(["event_name", "type_id"]).size().reset_index().sort_values(by=0, ascending=False)

,event_name,type_id,0
33,Pass,1,357793
32,Out,5,43096
3,Ball recovery,49,38864
4,Ball touch,61,25147
1,Aerial,44,20520
22,Foul,4,19680
16,Deleted event,43,19196
48,Take On,3,14683
10,Clearance,12,14096
0,50/50,67,13646


In [140]:
#Working on the Passes
passes = explode_event(match_events, 1, 0.15)
passes[["Pass End X","Pass End Y"]] = passes[["Pass End X","Pass End Y"]].astype(float)
passes["outcome"] = passes["outcome"].astype(int)
passes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 357793 entries, 0 to 357792
Data columns (total 32 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                357793 non-null  int64  
 1   event_id          357793 non-null  int64  
 2   type_id           357793 non-null  int64  
 3   event_name        357793 non-null  object 
 4   period_id         357793 non-null  int64  
 5   min               357793 non-null  int64  
 6   sec               357793 non-null  int64  
 7   team_id           357793 non-null  object 
 8   outcome           357793 non-null  int64  
 9   x                 357793 non-null  float64
 10  y                 357793 non-null  float64
 11  timestamp         357793 non-null  object 
 12  timestamp_utc     357793 non-null  object 
 13  last_modified     357793 non-null  object 
 14  version           357793 non-null  object 
 15  game_id           357793 non-null  object 
 16  player_id         35

# **Comparing Barcelona and Real Madrid’s Home Playing Style Through Passing Metrics**

In [141]:
#Find the Barcelona Id's at home
barcelona_id = all_pass[all_pass["home_team_name"] == "Barcelona"]["home_team_id"]
barcelona_id.unique()

array(['178'], dtype=object)

In [142]:
#Find the Real Madrid Id's at home
real_id = match_events[match_events["home_team_name"] == "Real Madrid"]["home_team_id"]
real_id.unique()

array(['186'], dtype=object)

In [179]:
#BARÇA AT HOME
#Total passes
barça_pass = passes[(passes["home_team_id"] == "178") & (passes["team_id"] == "178")]
#Accurate passes
barça_acc_pass = passes[(passes["home_team_id"] == "178") & (passes["team_id"] == "178") & (passes["outcome"] == 1)]
#Forward passes
barça_forward_pass = barça_pass[barça_pass["Pass End X"] > barça_pass["x"]]
#Forward accurate passes
barça_forward_acc_pass = barça_acc_pass[barça_acc_pass["Pass End X"] > barça_acc_pass["x"]]

In [149]:
#REAL AT HOME
#Total passes
real_pass = passes[(passes["home_team_id"] == "186") & (passes["team_id"] == "186")]
#Accurate passes
real_acc_pass = passes[(passes["home_team_id"] == "186") & (passes["team_id"] == "186") & (passes["outcome"] == 1)]
#Forward passes
real_forward_pass = real_pass[real_pass["Pass End X"] > real_pass["x"]]
#Forward accurate passes
real_forward_acc_pass = real_acc_pass[real_acc_pass["Pass End X"] > real_acc_pass["x"]]

***Check the results***

In [177]:
#Total Passes of both clubes

In [158]:
len(barça_pass)

13044

In [159]:
len(real_pass)

12918

In [ ]:
#Total Forward passes

In [160]:
len(barça_forward_pass)

7699

In [161]:
len(real_forward_pass)

7757

In [ ]:
#Percentage of successful passes

In [187]:
barça_acc = len(barça_acc_pass) / len(barça_pass)
round(barça_acc, 2)

0.87

In [188]:
real_acc = len(real_acc_pass) / len(real_pass)
round(real_acc, 2)

0.89

In [189]:
#Percentage of successful forward passes
barça_fwd_acc = len(barça_forward_acc_pass) / len(barça_forward_pass)
round(barça_fwd_acc, 2)

0.83

In [190]:
#Percentage of successful forward passes
real_fwd_acc = len(real_forward_acc_pass) / len(real_forward_pass)
round(real_fwd_acc, 2)

0.86

**CONCLUSIONS**
*   It seems that the style of play was similar
*   Barça's "tiki taka" does not exist anymore?